# Primer experimento RNN: F2 positiva y exportación reproducible

Este notebook entrena `rnn_v1_seed42` con el código compartido del repositorio. La RNN usa el paquete congelado `muestras_preprocesadas_v1_3`: train 2022–2023 y validación 2024. **2025 permanece reservada.** No se reajusta el preprocesador.

La selección de pesos usa F2 de validación con umbral fijo 0,5 y desempata por menor pérdida. Después, con esos pesos congelados, se busca el umbral de mayor F2 en 0,05–0,95. Se conservan los resultados de ambas decisiones.

El notebook se distribuye sin ejecuciones ni métricas. El entrenamiento, el modelo y la evaluación residen en módulos y comandos comunes; las celdas coordinan el flujo.

## 1. Preparar el paquete localmente

Desde la raíz del repositorio, ejecutar **antes de subir archivos a Colab**:

```bash
python scripts/empaquetar_experimento_rnn.py \
  --config configs/experimento_rnn_v1.json \
  --output /tmp/rnn_colab_v1.zip
```

Guardar el SHA-256 que muestre el empaquetador para contrastarlo aquí. Ese ZIP lleva código, configuración, dependencias y los artefactos necesarios de train/validación. Tras extraer, se exige `colab_package_manifest.json` versión `rnn_colab_inputs_v1`, `test_arrays_included=false`, conjunto exacto de archivos y SHA-256 correcto de cada archivo. No hay que cargar datos adicionales ni archivos de prueba 2025.

Abrir este notebook en Google Colab. Para usar GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. El dispositivo `auto` utiliza CUDA si está disponible y CPU en caso contrario. Google Drive es opcional.

Ejecutar las siguientes celdas en orden desde una sesión limpia. Si `/content/f1-rnn` ya existe, iniciar una sesión nueva para evitar mezclar corridas.

In [ ]:
from pathlib import Path
import hashlib
import json
import stat
import subprocess
import sys
import zipfile

from google.colab import files

# Opcional: copiar aquí el SHA-256 calculado localmente.
EXPECTED_PACKAGE_SHA256 = ""
ROOT = Path("/content/f1-rnn")
PACKAGE_PATH = Path("/content/rnn_colab_v1_uploaded.zip")

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Sube exactamente un archivo ZIP generado por el empaquetador.")
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
if Path(uploaded_name).suffix.lower() != ".zip":
    raise ValueError("El archivo debe ser un ZIP generado por el empaquetador.")
package_sha256 = hashlib.sha256(uploaded_bytes).hexdigest()
print("SHA-256 del paquete:", package_sha256)
if EXPECTED_PACKAGE_SHA256 and package_sha256 != EXPECTED_PACKAGE_SHA256.strip().lower():
    raise ValueError("El SHA-256 no coincide con el paquete local.")
if ROOT.exists():
    raise FileExistsError("/content/f1-rnn ya existe. Usa una sesión limpia para esta corrida.")
PACKAGE_PATH.write_bytes(uploaded_bytes)
del uploaded_bytes, uploaded

# Validar todos los nombres antes de extraer: sin rutas que salgan del destino,
# enlaces simbólicos ni destinos repetidos.
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    members = archive.infolist()
    destinations = set()
    for member in members:
        name = member.filename
        if "\\" in name:
            raise ValueError(f"Ruta ZIP no admitida: {name!r}")
        candidate = Path(name)
        if candidate.is_absolute() or ".." in candidate.parts:
            raise ValueError(f"Ruta ZIP fuera del destino: {name!r}")
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError(f"Enlace simbólico no admitido: {name!r}")
        destination = (ROOT / candidate).resolve()
        if not destination.is_relative_to(ROOT.resolve()) or destination == ROOT.resolve():
            raise ValueError(f"Ruta ZIP no admitida: {name!r}")
        if destination in destinations:
            raise ValueError(f"Destino ZIP repetido: {name!r}")
        destinations.add(destination)
    ROOT.mkdir(parents=True)
    archive.extractall(ROOT)

# Comprobar el contrato y todos los hashes del paquete antes de instalar o
# importar su código. El manifiesto no se incluye en su propio mapa de hashes.
package_manifest_path = ROOT / "colab_package_manifest.json"
package_manifest = json.loads(package_manifest_path.read_text(encoding="utf-8"))
if package_manifest.get("artifact_version") != "rnn_colab_inputs_v1":
    raise ValueError("Versión de paquete Colab no admitida.")
if package_manifest.get("test_arrays_included") is not False:
    raise ValueError("El paquete debe declarar test_arrays_included=false.")
file_hashes = package_manifest.get("files_sha256")
if not isinstance(file_hashes, dict) or not file_hashes:
    raise ValueError("El manifiesto debe enumerar los archivos y sus SHA-256.")
for name, expected_hash in file_hashes.items():
    if not isinstance(name, str) or "\\" in name:
        raise ValueError(f"Ruta de manifiesto no admitida: {name!r}")
    relative = Path(name)
    resolved = (ROOT / relative).resolve()
    if relative.is_absolute() or ".." in relative.parts or not resolved.is_relative_to(ROOT.resolve()):
        raise ValueError(f"Ruta de manifiesto fuera del destino: {name!r}")
    if resolved == ROOT.resolve() or resolved == package_manifest_path.resolve():
        raise ValueError(f"Ruta de manifiesto no admitida: {name!r}")
    if not isinstance(expected_hash, str) or len(expected_hash) != 64:
        raise ValueError(f"SHA-256 no válido para {name!r}")
    if any(character not in "0123456789abcdef" for character in expected_hash):
        raise ValueError(f"SHA-256 no válido para {name!r}")

expected_files = set(file_hashes) | {"colab_package_manifest.json"}
actual_files = {file.relative_to(ROOT).as_posix() for file in ROOT.rglob("*") if file.is_file()}
if actual_files != expected_files:
    raise ValueError(
        "Archivos distintos del manifiesto: "
        f"faltantes={sorted(expected_files - actual_files)}, "
        f"adicionales={sorted(actual_files - expected_files)}"
    )
for name, expected_hash in file_hashes.items():
    digest = hashlib.sha256()
    with (ROOT / name).open("rb") as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected_hash:
        raise ValueError(f"El SHA-256 no coincide para {name!r}")
print("Manifiesto y SHA-256 verificados:", len(file_hashes), "archivos; sin arrays de prueba.")

required = [
    "requirements-experiments.txt",
    "configs/experimento_rnn_v1.json",
    "scripts/entrenar_rnn.py",
    "scripts/evaluar_rnn.py",
    "src/f1_pitstop/rnn_model.py",
    "EDA/dataset/muestras_preprocesadas_v1_3/preprocessor.json",
    "EDA/dataset/muestras_preprocesadas_v1_3/manifest.json",
]
missing = [name for name in required if not (ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Paquete incompleto: {missing}")
print("Paquete extraído en", ROOT)


## 2. Instalar dependencias y revisar la configuración

La instalación respeta los rangos del archivo de requisitos y permite conservar un PyTorch CUDA compatible de Colab. No se fuerza un índice de wheels CPU. La exportación registra las versiones exactas empleadas; el hash del paquete identifica los archivos cargados.

Esta referencia usa una sola `RNNCell` tanh de 32 unidades, dos embeddings dinámicos de dimensión 4 y una rama estática mínima de 16 unidades. A los 22 valores numéricos se añaden explícitamente 22 bits de disponibilidad de sus máscaras. Los conteos y fracciones de cobertura de QA no son entradas.

La configuración fija semilla 42, Adam con learning rate 0,001, BCE sin ponderación inicial, batch 256, máximo 30 épocas, paciencia 6 y recorte de gradiente a 1. Cualquier cambio se registra como otro experimento.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-experiments.txt")],
    check=True,
)


In [ ]:
from importlib import metadata

sys.path.insert(0, str(ROOT / "src"))
import torch
from f1_pitstop.rnn_model import SmallPitStopRNN

CONFIG_PATH = ROOT / "configs/experimento_rnn_v1.json"
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
print(json.dumps(config, ensure_ascii=False, indent=2))
print("Código compartido:", SmallPitStopRNN.__module__ + "." + SmallPitStopRNN.__name__)
print("Python:", sys.version)
for dependency in ("torch", "numpy", "pandas"):
    print(dependency + ":", metadata.version(dependency))
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Se entrenará en CPU; esta ejecución puede tardar más.")


## 3. Entrenar mediante el comando compartido

El loader comprueba hashes, alineación, esquema y población supervisada antes de entrenar. Solo se consume train y validación. El preprocesador ya fue ajustado con 46.162 muestras elegibles de train; aquí se reutiliza su estado.

El checkpoint se elige con F2 positiva de validación a umbral **fijo 0,5**, con menor pérdida de validación como desempate. La paciencia observa ese mismo criterio. Una vez congelado el checkpoint, se selecciona el umbral en `0,05, 0,06, …, 0,95`; los empates se resuelven por mayor precision y luego mayor umbral. No se selecciona una época diferente tras buscar el umbral.

La salida debe ser nueva. El comando genera tanto la carpeta de artefactos como su ZIP hermano.

In [ ]:
RUN_DIR = ROOT / "runs/rnn_v1_seed42"
RUN_ZIP = RUN_DIR.with_suffix(".zip")
if RUN_DIR.exists() or RUN_ZIP.exists():
    raise FileExistsError("La salida de esta corrida ya existe. Conserva sus artefactos antes de repetir.")

subprocess.run(
    [
        sys.executable,
        "scripts/entrenar_rnn.py",
        "--config", "configs/experimento_rnn_v1.json",
        "--output", str(RUN_DIR.relative_to(ROOT)),
    ],
    cwd=ROOT,
    check=True,
)


## 4. Inspeccionar la evidencia

El historial permite revisar el aprendizaje y el checkpoint elegido. Las métricas incluyen el resultado a umbral 0,5 y a umbral seleccionado sobre las mismas probabilidades. La diferencia representa el efecto de cambiar la regla de alerta con pesos congelados.

Revisar precision, recall, F1, F2, matriz de confusión y cantidad de muestras elegibles. Estas métricas de entrenamiento se calculan por muestra piloto-vuelta. La cobertura por evento se calcula después, únicamente en el evaluador local, con las fuentes históricas verificadas por hash. El resultado ajustado en validación orienta desarrollo; no sustituye una evaluación final independiente. Una única semilla tampoco permite estimar variabilidad entre corridas.

In [ ]:
import pandas as pd
from IPython.display import display

history_path = RUN_DIR / "history.csv"
metrics_path = RUN_DIR / "metrics.json"
display(pd.read_csv(history_path))
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Artefactos:")
for artifact in sorted(RUN_DIR.iterdir()):
    if artifact.is_file():
        print(" -", artifact.name)


## 5. Descargar la corrida

El ZIP conserva pesos (`state_dict`), arquitectura, configuración, preprocesador, umbral, entorno exacto, historial, métricas, predicciones de validación por `sample_id` y manifiesto con hashes. Guardarlo antes de cerrar Colab.

La descarga utiliza el ZIP que generó el entrenamiento, sin reconstruir ni modificar sus artefactos.

In [ ]:
if not RUN_ZIP.is_file():
    raise FileNotFoundError(f"El entrenamiento no generó el ZIP esperado: {RUN_ZIP}")
print("Descarga:", RUN_ZIP)
files.download(str(RUN_ZIP))


### Copia opcional en Google Drive

Esta celda está desactivada por defecto. Para conservar una segunda copia en Drive, cambiar `COPY_TO_DRIVE` a `True`; Colab solicitará acceso al montar Drive. No hace falta para descargar el ZIP ni evaluar localmente.

In [ ]:
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    drive_directory = Path("/content/drive/MyDrive/F1-PitStop-Predictor")
    drive_directory.mkdir(parents=True, exist_ok=True)
    drive_target = drive_directory / RUN_ZIP.name
    if drive_target.exists():
        raise FileExistsError(f"Ya existe una copia: {drive_target}")
    shutil.copy2(RUN_ZIP, drive_target)
    print("Copia conservada en", drive_target)


## 6. Volver al repositorio y evaluar en CPU

Extraer el ZIP descargado. Desde la raíz del repositorio, usar como `--bundle` la **carpeta que contiene `weights.pt`**, no el ZIP:

```bash
python scripts/evaluar_rnn.py \
  --bundle RUTA_DIRECTORIO_MODELO \
  --data EDA/dataset/muestras_preprocesadas_v1_3 \
  --output eval/rnn_v1_validation \
  --device cpu
```

El evaluador local también necesita las fuentes que ya existen en el repositorio: `events.csv` de la publicación `etiquetado_pit_entry_v3_2022_2025` y `sample_index.csv.gz` de `muestras_causales_v1_3`, en las rutas fijadas en la configuración. Ambas se verifican por hash y **no se incluyen en el paquete Colab**. Se usan para calcular la cobertura por evento de validación, únicamente en esta evaluación local.

El evaluador carga el mismo modelo, pesos, preprocesador y umbral congelados; contrasta probabilidades de validación por `sample_id` con las exportadas por Colab. Usa `parity_atol=1e-5` y `parity_rtol=1e-4`, registradas en configuración y artefactos, sin exigir igualdad bit a bit entre GPU y CPU. Si la comparación falla, se investiga la discrepancia antes de publicar métricas. Scores muy próximos al umbral pueden cambiar de lado dentro de la tolerancia; el evaluador registra las diferencias y cuenta esos cambios de decisión.

Para la siguiente iteración, conservar esta corrida y declarar el factor que se modifica: capacidad, entradas, entrenamiento o decisión. Comparar umbrales mantiene pesos congelados; comparar aprendizaje mantiene el protocolo de selección. **2025 sigue reservada para la evaluación final acordada.**

Contrato completo: `docs/specs/11_experimento_rnn_inicial.md`.